# Data Foundation: MovieLens + M3L + Wikidata

This notebook builds the project’s movie-level data foundation from the actual source families used in the study:

1. **MovieLens 20M** for movie metadata, ratings, tags and IMDb/TMDb bridge identifiers.
2. **M3L-20M / Binge Watch** for recommender interactions and multimodal feature availability.
3. **Wikidata Query Service** for country, original-language and production-company metadata.

The point of this notebook is transparency. Later notebooks can use prepared tables, but this notebook shows where the data comes from, how identifiers are joined, how Wikidata enrichment works, and where metadata coverage is strong or weak. No synthetic data is generated.


## 1. Source Map

The project does **not** treat generated CSV tables as primary sources. They are derived artifacts. The actual data basis is the small set of sources below.

| Layer | Primary source | Why we need it | Key fields |
|---|---|---|---|
| Catalogue and user behaviour | MovieLens 20M | titles, genres, ratings, tags and IMDb bridge | `movieId`, `title`, `genres`, `rating`, `imdbId`, `tmdbId` |
| Recommender interaction space | M3L-20M / Binge Watch | train/valid/test interactions and multimodal feature rows | `userID`, `itemID`, `rating`, `x_label`, MPNet / CLIP feature files |
| Cultural metadata | Wikidata via SPARQL | production country, original language, production company and release date | IMDb property `P345`, country `P495`, language `P364`, production company `P272` |

Derived helper tables such as the M3L-to-MovieLens bridge and the Wikidata cache are created only to make the joins reproducible and polite to external services.


In [1]:
from pathlib import Path
import ast
import json
import os
import re
import time
from collections import defaultdict

import numpy as np
import pandas as pd
import requests
from IPython.display import Markdown, display
from tqdm.auto import tqdm

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_colwidth", 120)

NO_SYNTHETIC_DATA = True
SEED = 42

# Keep the first run polite and reproducible. If a Wikidata cache already exists,
# the notebook uses it. If no cache exists, this flag allows a real SPARQL rebuild.
QUERY_WIKIDATA_IF_CACHE_MISSING = True
WIKIDATA_BATCH_SIZE = 200
WIKIDATA_SLEEP_SECONDS = 0.08

# Set to an integer for a quick local demo, or None for all MovieLens ratings.
RATING_NROWS = 1_000_000

# Notebook execution may start either from the repo root or from /notebooks.
cwd = Path.cwd().resolve()
PROJECT_ROOT = cwd.parent if cwd.name == "notebooks" else cwd
DATA_INTERIM = PROJECT_ROOT / "data" / "interim"
DATA_PROCESSED = PROJECT_ROOT / "data" / "processed"
OUTPUTS = PROJECT_ROOT / "outputs"

for folder in [DATA_INTERIM, DATA_PROCESSED, OUTPUTS]:
    folder.mkdir(parents=True, exist_ok=True)

# During local development the raw course data may sit one or two folders above
# the submission repository. Public users can instead put files under the repo.
source_roots = []
if os.getenv("DSAI2_DATA_ROOT"):
    source_roots.append(Path(os.environ["DSAI2_DATA_ROOT"]).expanduser().resolve())
source_roots.extend([PROJECT_ROOT, PROJECT_ROOT / "data" / "raw"])
source_roots.extend([p for p in PROJECT_ROOT.parents[:3]])

# Preserve order while removing duplicates.
SOURCE_ROOTS = []
for root in source_roots:
    if root not in SOURCE_ROOTS:
        SOURCE_ROOTS.append(root)

print(f"Configured {len(SOURCE_ROOTS)} source-root search locations. Paths are displayed as repo/... or local_data_root/... in the inventory below.")


Configured 5 source-root search locations. Paths are displayed as repo/... or local_data_root/... in the inventory below.


## 2. Locate Raw Sources

This cell searches for the raw/local source files. It deliberately distinguishes between **primary data sources** and **derived caches**. A missing required file stops the notebook rather than silently creating fake data.


In [2]:
def find_existing(relative_candidates, roots=SOURCE_ROOTS):
    """Return the first existing file/folder from a list of repo-relative candidates."""
    for root in roots:
        for rel in relative_candidates:
            path = root / rel
            if path.exists():
                return path
    return None


def is_git_lfs_pointer(path: Path) -> bool:
    """M3L checkouts sometimes contain tiny Git-LFS pointer files instead of data."""
    if not path or not path.exists() or not path.is_file():
        return False
    with path.open("rb") as handle:
        return handle.read(128).startswith(b"version https://git-lfs.github.com/spec/v1")


def display_path(path):
    """Show reproducible-looking paths instead of local absolute user paths."""
    if not path:
        return ""
    for root in SOURCE_ROOTS:
        try:
            rel = path.relative_to(root)
        except ValueError:
            continue
        prefix = "repo" if root == PROJECT_ROOT else "local_data_root"
        return str(Path(prefix) / rel)
    return path.name


source_specs = [
    {
        "name": "MovieLens movies",
        "kind": "primary source",
        "role": "Movie titles and genres",
        "required": True,
        "candidates": [
            Path("MovieLens 20M Dataset/movie.csv"),
            Path("MovieLens 20M Dataset/movies.csv"),
            Path("data/archive/movie.csv"),
            Path("archive/movie.csv"),
            Path("ml-20m/movies.csv"),
            Path("data/raw/ml-20m/movies.csv"),
        ],
    },
    {
        "name": "MovieLens links",
        "kind": "primary source",
        "role": "IMDb/TMDb bridge",
        "required": True,
        "candidates": [
            Path("MovieLens 20M Dataset/link.csv"),
            Path("MovieLens 20M Dataset/links.csv"),
            Path("data/archive/link.csv"),
            Path("archive/link.csv"),
            Path("ml-20m/links.csv"),
            Path("data/raw/ml-20m/links.csv"),
        ],
    },
    {
        "name": "MovieLens ratings",
        "kind": "primary source",
        "role": "Ratings for popularity and user-interest context",
        "required": True,
        "candidates": [
            Path("MovieLens 20M Dataset/rating.csv"),
            Path("MovieLens 20M Dataset/ratings.csv"),
            Path("data/archive/rating.csv"),
            Path("archive/rating.csv"),
            Path("ml-20m/ratings.csv"),
            Path("data/raw/ml-20m/ratings.csv"),
        ],
    },
    {
        "name": "M3L interactions",
        "kind": "primary source",
        "role": "Train/validation/test recommender interactions",
        "required": True,
        "candidates": [Path("m3l-20m/m3l-20m.inter"), Path("data/m3l-20m/m3l-20m.inter")],
    },
    {
        "name": "MPNet matrix",
        "kind": "primary source feature",
        "role": "M3L text embeddings used by text-content model and ID bridge check",
        "required": False,
        "candidates": [Path("m3l-20m/text/mpnet.npy"), Path("data/m3l-20m/text/mpnet.npy")],
    },
    {
        "name": "CLIP image matrix",
        "kind": "primary source feature",
        "role": "M3L poster embeddings used by image-content model",
        "required": False,
        "candidates": [Path("m3l-20m/image/clip_image.npy"), Path("data/m3l-20m/image/clip_image.npy")],
    },
    {
        "name": "MPNet JSON folder",
        "kind": "primary source feature",
        "role": "Per-movie MPNet JSON files; used to reconstruct M3L item_id to MovieLens movieId bridge if needed",
        "required": False,
        "candidates": [Path("TEXT_mpnet"), Path("data/raw/TEXT_mpnet"), Path("features_json_format/1_text_feat/mpnet")],
    },
    {
        "name": "CLIP image JSON folder",
        "kind": "primary source feature",
        "role": "Per-movie CLIP-image JSON files for feature coverage",
        "required": False,
        "candidates": [Path("IMG_clip-image"), Path("data/raw/IMG_clip-image"), Path("features_json_format/2_image_feat/clip-image")],
    },
    {
        "name": "M3L to MovieLens bridge",
        "kind": "derived join cache",
        "role": "Maps M3L internal item_id values to MovieLens movieId values",
        "required": False,
        "candidates": [Path("data/interim/m3l_internal_to_movielens.csv")],
    },
    {
        "name": "Wikidata cultural metadata cache",
        "kind": "derived enrichment cache",
        "role": "Cached SPARQL results for country, language and production metadata",
        "required": False,
        "candidates": [Path("data/interim/wikidata_movie_metadata_extended.csv"), Path("data/interim/wikidata_movie_metadata.csv")],
    },
]

resolved = {}
rows = []
for spec in source_specs:
    path = find_existing(spec["candidates"])
    resolved[spec["name"]] = path
    rows.append({
        "name": spec["name"],
        "kind": spec["kind"],
        "role": spec["role"],
        "required": spec["required"],
        "resolved_path": display_path(path),
        "exists": bool(path),
        "size_mb": round(path.stat().st_size / 1024**2, 2) if path and path.is_file() else np.nan,
        "is_lfs_pointer": is_git_lfs_pointer(path) if path else False,
    })

source_inventory = pd.DataFrame(rows)
display(source_inventory)

missing_required = source_inventory.query("required and (not exists or is_lfs_pointer)")
if len(missing_required):
    display(missing_required)
    raise FileNotFoundError("Missing required primary data files. No synthetic fallback is allowed.")


,name,kind,role,required,resolved_path,exists,size_mb,is_lfs_pointer
0,MovieLens movies,primary source,Movie titles and genres,True,local_data_root/MovieLens 20M Dataset/movie.csv,True,1.42,False
1,MovieLens links,primary source,IMDb/TMDb bridge,True,local_data_root/MovieLens 20M Dataset/link.csv,True,0.51,False
2,MovieLens ratings,primary source,Ratings for popularity and user-interest context,True,local_data_root/MovieLens 20M Dataset/rating.csv,True,658.37,False
3,M3L interactions,primary source,Train/validation/test recommender interactions,True,local_data_root/m3l-20m/m3l-20m.inter,True,265.74,False
4,MPNet matrix,primary source feature,M3L text embeddings used by text-content model and ID bridge check,False,local_data_root/m3l-20m/text/mpnet.npy,True,111.38,False
5,CLIP image matrix,primary source feature,M3L poster embeddings used by image-content model,False,local_data_root/m3l-20m/image/clip_image.npy,True,74.25,False
6,MPNet JSON folder,primary source feature,Per-movie MPNet JSON files; used to reconstruct M3L item_id to MovieLens movieId bridge if needed,False,local_data_root/TEXT_mpnet,True,NaN,False
7,CLIP image JSON folder,primary source feature,Per-movie CLIP-image JSON files for feature coverage,False,local_data_root/IMG_clip-image,True,NaN,False
8,M3L to MovieLens bridge,derived join cache,Maps M3L internal item_id values to MovieLens movieId values,False,local_data_root/data/interim/m3l_internal_to_movielens.csv,True,0.20,False
9,Wikidata cultural metadata cache,derived enrichment cache,"Cached SPARQL results for country, language and production metadata",False,local_data_root/data/interim/wikidata_movie_metadata_extended.csv,True,2.55,False


## 3. Load MovieLens and M3L Interactions

MovieLens gives us human-readable movie metadata and IMDb identifiers. M3L gives us the recommender interaction space. The two datasets do not use the same item identifier internally, so the bridge is handled explicitly in the next section.


In [3]:
def parse_title_year(title):
    """Split MovieLens titles like 'Amelie (Fabuleux destin...) (2001)' into title and year."""
    if pd.isna(title):
        return pd.Series({"clean_title": pd.NA, "year": pd.NA})
    title = str(title)
    match = re.search(r"\((\d{4})\)\s*$", title)
    year = int(match.group(1)) if match else pd.NA
    clean_title = re.sub(r"\s*\(\d{4}\)\s*$", "", title).strip()
    return pd.Series({"clean_title": clean_title, "year": year})


def imdb_numeric_to_tt(value):
    if pd.isna(value):
        return pd.NA
    return "tt" + str(int(value)).zfill(7)


movies = pd.read_csv(resolved["MovieLens movies"])
links = pd.read_csv(resolved["MovieLens links"])
ratings = pd.read_csv(resolved["MovieLens ratings"], nrows=RATING_NROWS)

movies = movies.join(movies["title"].apply(parse_title_year))
links["imdb_id_str"] = links["imdbId"].apply(imdb_numeric_to_tt)

m3l_interactions = pd.read_csv(
    resolved["M3L interactions"],
    sep="\t",
    dtype={"userID": "int32", "itemID": "int32", "rating": "float32", "x_label": "int8"},
).rename(columns={"userID": "user_id", "itemID": "item_id", "rating": "rating_or_weight", "x_label": "split"})

m3l_interactions["split"] = m3l_interactions["split"].map({0: "train", 1: "valid", 2: "test"}).fillna(m3l_interactions["split"].astype(str))

source_counts = pd.DataFrame([
    {"object": "MovieLens movies", "rows": len(movies), "unique_ids": movies["movieId"].nunique()},
    {"object": "MovieLens links", "rows": len(links), "unique_ids": links["movieId"].nunique()},
    {"object": "Loaded MovieLens ratings", "rows": len(ratings), "unique_ids": ratings["movieId"].nunique()},
    {"object": "M3L interactions", "rows": len(m3l_interactions), "unique_ids": m3l_interactions["item_id"].nunique()},
    {"object": "M3L users", "rows": m3l_interactions["user_id"].nunique(), "unique_ids": m3l_interactions["user_id"].nunique()},
])

display(source_counts)
display(movies.head())
display(m3l_interactions["split"].value_counts().rename_axis("split").reset_index(name="rows"))

display(Markdown(
    f"**Interpretation.** The notebook is now starting from primary inputs: {len(movies):,} MovieLens movie rows "
    f"and {len(m3l_interactions):,} M3L interaction rows. The MovieLens rating file is loaded with "
    f"`RATING_NROWS={RATING_NROWS}` for descriptive context; set it to `None` for a full local rebuild."
))


,object,rows,unique_ids
0,MovieLens movies,27278,27278
1,MovieLens links,27278,27278
2,Loaded MovieLens ratings,1000000,13950
3,M3L interactions,18777965,19009
4,M3L users,138493,138493


,movieId,title,genres,clean_title,year
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy,Toy Story,1995
1,2,Jumanji (1995),Adventure|Children|Fantasy,Jumanji,1995
2,3,Grumpier Old Men (1995),Comedy|Romance,Grumpier Old Men,1995
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance,Waiting to Exhale,1995
4,5,Father of the Bride Part II (1995),Comedy,Father of the Bride Part II,1995


,split,rows
0,train,15022372
1,test,1877797
2,valid,1877796


**Interpretation.** The notebook is now starting from primary inputs: 27,278 MovieLens movie rows and 18,777,965 M3L interaction rows. The MovieLens rating file is loaded with `RATING_NROWS=1000000` for descriptive context; set it to `None` for a full local rebuild.

## 4. Build or Load the M3L to MovieLens Bridge

M3L stores interactions by an internal `item_id`; MovieLens stores metadata by `movieId`. We therefore need a bridge. If the bridge cache exists, we load it. If it does not exist, we reconstruct it from the MPNet feature matrix and the per-movie MPNet JSON files.

This bridge is **not** a new data source. It is a reproducibility artifact that connects M3L rows back to MovieLens movie identifiers.


In [4]:
def vector_fingerprint(vector):
    """Create a compact deterministic fingerprint for matching matrix rows to JSON vectors."""
    arr = np.asarray(vector, dtype=np.float64)
    idx = [0, 1, 2, 3, 4, 5, 10, 50, 100, 200, 400, min(len(arr) - 1, 767)]
    return tuple(np.round(arr[idx], 8))


def build_internal_to_movielens_mapping(cache_path=None):
    """Load or reconstruct M3L internal item_id -> MovieLens movieId mapping."""
    cache_path = cache_path or (DATA_INTERIM / "m3l_internal_to_movielens.csv")
    if resolved["M3L to MovieLens bridge"] and resolved["M3L to MovieLens bridge"].exists():
        return pd.read_csv(resolved["M3L to MovieLens bridge"])
    if cache_path.exists():
        return pd.read_csv(cache_path)

    mpnet_matrix_path = resolved["MPNet matrix"]
    mpnet_json_dir = resolved["MPNet JSON folder"]
    if not mpnet_matrix_path or not mpnet_json_dir:
        raise FileNotFoundError(
            "Cannot reconstruct M3L item bridge. Provide data/interim/m3l_internal_to_movielens.csv "
            "or both m3l-20m/text/mpnet.npy and TEXT_mpnet JSON files."
        )

    matrix = np.load(mpnet_matrix_path, mmap_mode="r")
    lookup = {}
    collisions = defaultdict(list)
    for path in tqdm(sorted(mpnet_json_dir.glob("*.json")), desc="Indexing MPNet JSON vectors"):
        with path.open("r") as handle:
            obj = json.load(handle)
        movie_id_str = next(iter(obj))
        fingerprint = vector_fingerprint(obj[movie_id_str])
        if fingerprint in lookup:
            collisions[fingerprint].append(int(movie_id_str))
        else:
            lookup[fingerprint] = int(movie_id_str)

    rows, misses = [], []
    for internal_item_id in tqdm(range(matrix.shape[0]), desc="Matching M3L internal item IDs"):
        movie_id = lookup.get(vector_fingerprint(matrix[internal_item_id]))
        if movie_id is None:
            misses.append(internal_item_id)
        else:
            rows.append({"item_id": internal_item_id, "movieId": movie_id})

    if misses:
        raise ValueError(f"Could not map {len(misses)} internal item IDs. First misses: {misses[:10]}")

    mapping = pd.DataFrame(rows)
    mapping.to_csv(cache_path, index=False)
    return mapping


item_id_map = build_internal_to_movielens_mapping()
bridge_diagnostics = pd.DataFrame([
    {"metric": "bridge rows", "value": len(item_id_map)},
    {"metric": "unique M3L item_id", "value": item_id_map["item_id"].nunique()},
    {"metric": "unique MovieLens movieId", "value": item_id_map["movieId"].nunique()},
    {"metric": "duplicate MovieLens movieId rows", "value": int(item_id_map.duplicated("movieId").sum())},
])

display(bridge_diagnostics)
display(item_id_map.head())

display(Markdown(
    f"**Interpretation.** The bridge maps {len(item_id_map):,} M3L internal items to "
    f"{item_id_map['movieId'].nunique():,} MovieLens movie IDs. Duplicate `movieId` rows are retained as a diagnostic, "
    "but later movie-level tables keep one row per MovieLens movie."
))


,metric,value
0,bridge rows,19009
1,unique M3L item_id,19009
2,unique MovieLens movieId,18973
3,duplicate MovieLens movieId rows,36


,item_id,movieId
0,0,1
1,1,3
2,2,4
3,3,5
4,4,7


**Interpretation.** The bridge maps 19,009 M3L internal items to 18,973 MovieLens movie IDs. Duplicate `movieId` rows are retained as a diagnostic, but later movie-level tables keep one row per MovieLens movie.

## 5. Wikidata Enrichment via SPARQL

MovieLens does not contain production country or original-language metadata. We enrich films through IMDb identifiers using Wikidata property `P345`. The query also retrieves:

- `P495`: country of origin / production country proxy
- `P364`: original language of film or TV show
- `P407`: language of work or name, used as a secondary language signal
- `P577`: publication date
- `P272`: production company, plus company country / headquarters country where available

The cache exists to avoid hammering Wikidata on every run. If the cache is missing, the same cell can rebuild it with real SPARQL requests.


In [5]:
WIKIDATA_ENDPOINT = "https://query.wikidata.org/sparql"


def query_wikidata_extended(imdb_ids, batch_size=WIKIDATA_BATCH_SIZE, sleep=WIKIDATA_SLEEP_SECONDS):
    """Query Wikidata by IMDb IDs and return one aggregated row per matched film."""
    rows = []
    imdb_ids = [x for x in imdb_ids if isinstance(x, str) and x.startswith("tt")]

    for start in tqdm(range(0, len(imdb_ids), batch_size), desc="Querying Wikidata"):
        batch = imdb_ids[start:start + batch_size]
        values = " ".join(f'"{imdb}"' for imdb in batch)
        query = f"""
        SELECT ?imdb ?film ?filmLabel
          (GROUP_CONCAT(DISTINCT ?countryLabel; separator="|") AS ?countries)
          (GROUP_CONCAT(DISTINCT ?originalLanguageLabel; separator="|") AS ?originalLanguages)
          (GROUP_CONCAT(DISTINCT ?workLanguageLabel; separator="|") AS ?workLanguages)
          (SAMPLE(?publicationDate) AS ?publicationDate)
          (GROUP_CONCAT(DISTINCT ?productionCompanyLabel; separator="|") AS ?productionCompanies)
          (GROUP_CONCAT(DISTINCT ?companyCountryLabel; separator="|") AS ?productionCompanyCountries)
          (GROUP_CONCAT(DISTINCT ?companyHqCountryLabel; separator="|") AS ?productionCompanyHqCountries)
        WHERE {{
          VALUES ?imdb {{ {values} }}
          ?film wdt:P345 ?imdb .
          OPTIONAL {{ ?film wdt:P495 ?country . }}
          OPTIONAL {{ ?film wdt:P364 ?originalLanguage . }}
          OPTIONAL {{ ?film wdt:P407 ?workLanguage . }}
          OPTIONAL {{ ?film wdt:P577 ?publicationDate . }}
          OPTIONAL {{
            ?film wdt:P272 ?productionCompany .
            OPTIONAL {{ ?productionCompany wdt:P17 ?companyCountry . }}
            OPTIONAL {{ ?productionCompany wdt:P159/wdt:P17 ?companyHqCountry . }}
          }}
          SERVICE wikibase:label {{ bd:serviceParam wikibase:language "en".
            ?film rdfs:label ?filmLabel .
            ?country rdfs:label ?countryLabel .
            ?originalLanguage rdfs:label ?originalLanguageLabel .
            ?workLanguage rdfs:label ?workLanguageLabel .
            ?productionCompany rdfs:label ?productionCompanyLabel .
            ?companyCountry rdfs:label ?companyCountryLabel .
            ?companyHqCountry rdfs:label ?companyHqCountryLabel .
          }}
        }}
        GROUP BY ?imdb ?film ?filmLabel
        """
        try:
            response = requests.get(
                WIKIDATA_ENDPOINT,
                params={"query": query, "format": "json"},
                headers={"User-Agent": "WU-Data-Algorithmic-Governance-Student-Project/1.0"},
                timeout=90,
            )
            response.raise_for_status()
            bindings = response.json()["results"]["bindings"]
        except Exception as exc:
            print(f"Wikidata batch failed at {start}: {exc}")
            time.sleep(3)
            continue

        for row in bindings:
            rows.append({
                "imdb_id_str": row.get("imdb", {}).get("value"),
                "wikidata_uri": row.get("film", {}).get("value"),
                "title_wikidata": row.get("filmLabel", {}).get("value"),
                "country": row.get("countries", {}).get("value"),
                "original_language": row.get("originalLanguages", {}).get("value"),
                "language_of_work": row.get("workLanguages", {}).get("value"),
                "publication_date": row.get("publicationDate", {}).get("value"),
                "production_company": row.get("productionCompanies", {}).get("value"),
                "production_company_country": row.get("productionCompanyCountries", {}).get("value"),
                "production_company_hq_country": row.get("companyHqCountry", {}).get("value"),
            })
        time.sleep(sleep)
    return pd.DataFrame(rows)


def load_or_query_wikidata(links_df):
    """Use a local cache when present; otherwise rebuild via SPARQL."""
    cache_path = DATA_INTERIM / "wikidata_movie_metadata_extended.csv"
    existing_cache = resolved["Wikidata cultural metadata cache"] or (cache_path if cache_path.exists() else None)
    if existing_cache:
        wikidata_df = pd.read_csv(existing_cache)
        cache_source = display_path(existing_cache)
    else:
        if not QUERY_WIKIDATA_IF_CACHE_MISSING:
            raise FileNotFoundError(
                "Wikidata cache missing. Set QUERY_WIKIDATA_IF_CACHE_MISSING=True to rebuild via SPARQL."
            )
        imdb_ids = links_df["imdb_id_str"].dropna().drop_duplicates().tolist()
        wikidata_df = query_wikidata_extended(imdb_ids)
        wikidata_df.to_csv(cache_path, index=False)
        cache_source = display_path(cache_path)
    return wikidata_df, cache_source


wikidata_raw, wikidata_cache_source = load_or_query_wikidata(links)

display(wikidata_raw.head())
display(pd.DataFrame([{
    "cache_source": wikidata_cache_source,
    "rows": len(wikidata_raw),
    "unique_imdb": wikidata_raw["imdb_id_str"].nunique() if "imdb_id_str" in wikidata_raw else np.nan,
    "has_country_rows": wikidata_raw["country"].notna().sum() if "country" in wikidata_raw else np.nan,
    "has_language_rows": wikidata_raw["original_language"].notna().sum() if "original_language" in wikidata_raw else np.nan,
}]))


,imdb_id_str,wikidata_uri,title_wikidata,country,original_language,language_of_work,publication_date,production_company,production_company_country,production_company_hq_country
0,tt0114808,http://www.wikidata.org/entity/Q586569,The Star Maker,Italy,Italian,NaN,1996-07-11T00:00:00Z,Sciarló|Cecchi Gori Group,Italy,Italy
1,tt0112682,http://www.wikidata.org/entity/Q749988,The City of Lost Children,Germany|France|Spain,French,NaN,2023-10-05T00:00:00Z,Elías Querejeta PC|Tele München Group|StudioCanal|France 3 Cinéma|LEONINE Production|Constellation Productions|Q1347...,Spain|Germany|United Kingdom|France,Spain|Germany|France
2,tt0114709,http://www.wikidata.org/entity/Q171048,Toy Story,United States,English|Spanish,NaN,1996-03-21T00:00:00Z,Pixar|Walt Disney Pictures,United States,United States
3,tt0114746,http://www.wikidata.org/entity/Q175038,12 Monkeys,United States,English,NaN,1996-03-21T00:00:00Z,Universal Pictures,United States,United States
4,tt0114369,http://www.wikidata.org/entity/Q190908,Seven,United States,English,NaN,1995-11-23T00:00:00Z,New Line Cinema,United States,United States


,cache_source,rows,unique_imdb,has_country_rows,has_language_rows
0,local_data_root/data/interim/wikidata_movie_metadata_extended.csv,18684,18684,18679,18353


## 6. Aggregate Cultural Metadata and Build Audit Labels

Wikidata can return multiple values per film. We keep those values as lists rather than forcing a single country or language. This matters for co-productions and multilingual films.

The labels below are operational audit proxies:

- `is_european_wide`: at least one production-country proxy is in the wider Europe list.
- `is_non_english`: original-language metadata exists and does not include English.
- `is_us_origin`: at least one country proxy is the United States.
- `has_us_company_signal`: at least one production-company country or headquarters country is the United States.
- `is_long_tail`: item popularity is in the lower 20% by training interaction count.


In [6]:
WIDE_EUROPE = {
    "Austria", "Germany", "France", "Italy", "Spain", "Portugal", "Netherlands",
    "Belgium", "Luxembourg", "Ireland", "Denmark", "Sweden", "Finland",
    "Poland", "Czech Republic", "Czechia", "Slovakia", "Hungary", "Slovenia",
    "Croatia", "Romania", "Bulgaria", "Greece", "Cyprus", "Malta", "Estonia",
    "Latvia", "Lithuania", "Norway", "Switzerland", "Iceland", "United Kingdom",
    "Ukraine", "Serbia", "Bosnia and Herzegovina", "Montenegro", "North Macedonia",
    "Albania", "Moldova", "Russia", "Turkey",
}


def split_values(value):
    """Convert cached pipe-separated strings or list-like strings into clean lists."""
    if isinstance(value, list):
        return [str(x) for x in value if str(x) and str(x).lower() != "nan"]
    if pd.isna(value) or value == "":
        return []
    text = str(value)
    try:
        parsed = ast.literal_eval(text)
        if isinstance(parsed, list):
            return [str(x) for x in parsed if str(x) and str(x).lower() != "nan"]
    except Exception:
        pass
    if "|" in text:
        return sorted({part.strip() for part in text.split("|") if part.strip()})
    return [text]


def aggregate_wikidata(wikidata_df):
    """Return one row per IMDb ID with list-valued cultural metadata."""
    if wikidata_df.empty:
        return pd.DataFrame(columns=[
            "imdb_id_str", "wikidata_uri", "title_wikidata", "country", "original_language",
            "language_of_work", "publication_date", "production_company",
            "production_company_country", "production_company_hq_country",
        ])

    work = wikidata_df.copy()
    for col in [
        "country", "original_language", "language_of_work", "production_company",
        "production_company_country", "production_company_hq_country",
    ]:
        if col not in work:
            work[col] = pd.NA
        work[col] = work[col].apply(split_values)

    def merge_lists(series):
        merged = []
        for values in series:
            merged.extend(values if isinstance(values, list) else split_values(values))
        return sorted(set(merged))

    return work.groupby("imdb_id_str", dropna=False).agg({
        "wikidata_uri": "first",
        "title_wikidata": "first",
        "country": merge_lists,
        "original_language": merge_lists,
        "language_of_work": merge_lists,
        "publication_date": "first",
        "production_company": merge_lists,
        "production_company_country": merge_lists,
        "production_company_hq_country": merge_lists,
    }).reset_index()


train_interactions = m3l_interactions[m3l_interactions["split"].eq("train")]
item_counts = train_interactions.groupby("item_id").size().rename("train_interaction_count").reset_index()
wd_agg = aggregate_wikidata(wikidata_raw)

movie_level = (
    item_id_map.sort_values("item_id").drop_duplicates("movieId", keep="first")
    .merge(movies, on="movieId", how="left")
    .merge(links[["movieId", "imdbId", "tmdbId", "imdb_id_str"]], on="movieId", how="left")
    .merge(item_counts, on="item_id", how="left")
    .merge(wd_agg, on="imdb_id_str", how="left")
)

movie_level["train_interaction_count"] = movie_level["train_interaction_count"].fillna(0).astype(int)
q20 = movie_level["train_interaction_count"].quantile(0.20)
q80 = movie_level["train_interaction_count"].quantile(0.80)

movie_level["has_wikidata_match"] = movie_level["wikidata_uri"].notna()
movie_level["has_country"] = movie_level["country"].apply(lambda xs: isinstance(xs, list) and len(xs) > 0)
movie_level["has_original_language"] = movie_level["original_language"].apply(lambda xs: isinstance(xs, list) and len(xs) > 0)
movie_level["is_european_wide"] = movie_level["country"].apply(lambda xs: any(x in WIDE_EUROPE for x in xs) if isinstance(xs, list) else False)
movie_level["is_non_english"] = movie_level["original_language"].apply(lambda xs: bool(xs) and "English" not in xs if isinstance(xs, list) else pd.NA)
movie_level["is_us_origin"] = movie_level["country"].apply(lambda xs: "United States" in xs if isinstance(xs, list) else False)

def contains_us_company_signal(row):
    """Return True when company country or headquarters metadata points to the US."""
    company_country = row.get("production_company_country")
    company_hq_country = row.get("production_company_hq_country")
    values = []
    values.extend(company_country if isinstance(company_country, list) else [])
    values.extend(company_hq_country if isinstance(company_hq_country, list) else [])
    return "United States" in values

movie_level["has_us_company_signal"] = movie_level.apply(contains_us_company_signal, axis=1)
movie_level["is_long_tail"] = movie_level["train_interaction_count"] <= q20
movie_level["is_blockbuster_head"] = movie_level["train_interaction_count"] >= q80

preview_cols = [
    "item_id", "movieId", "title", "imdb_id_str", "country", "original_language",
    "production_company_country", "is_european_wide", "is_non_english",
    "is_us_origin", "has_us_company_signal", "is_long_tail", "train_interaction_count",
]

display(movie_level[preview_cols].head(12))

display(Markdown(
    "**Interpretation.** Country and language are now visible as explicit Wikidata-derived fields, not hidden inside precomputed tables. "
    "The labels remain proxies: co-productions can belong to multiple countries, and original language is not the same as dubbing, subtitles or audience language."
))


,item_id,movieId,title,imdb_id_str,country,original_language,production_company_country,is_european_wide,is_non_english,is_us_origin,has_us_company_signal,is_long_tail,train_interaction_count
0,0,1,Toy Story (1995),tt0114709,[United States],"[English, Spanish]",[United States],False,False,True,True,False,39948
1,1,3,Grumpier Old Men (1995),tt0113228,[United States],[English],[],False,False,True,False,False,10153
2,2,4,Waiting to Exhale (1995),tt0114885,[United States],[English],[United States],False,False,True,True,False,2164
3,3,5,Father of the Bride Part II (1995),tt0113041,[United States],[English],[United States],False,False,True,True,False,9736
4,4,7,Sabrina (1995),tt0114319,[United States],[English],[United States],False,False,True,True,False,10349
5,5,9,Sudden Death (1995),tt0114576,[United States],[English],[United States],False,False,True,True,False,3158
6,6,10,GoldenEye (1995),tt0113189,"[United Kingdom, United States]",[English],"[United Kingdom, United States]",True,False,True,True,False,23158
7,7,11,"American President, The (1995)",tt0112346,[United States],[English],[United States],False,False,True,True,False,14476
8,8,12,Dracula: Dead and Loving It (1995),tt0112896,"[France, Italy, United States]","[English, German]","[United Kingdom, United States]",True,False,True,True,False,3086
9,9,13,Balto (1995),tt0112453,[United States],[English],"[United Kingdom, United States]",False,False,True,True,False,1178


**Interpretation.** Country and language are now visible as explicit Wikidata-derived fields, not hidden inside precomputed tables. The labels remain proxies: co-productions can belong to multiple countries, and original language is not the same as dubbing, subtitles or audience language.

## 7. Feature Coverage and Join Funnel

Before any recommender result is interpreted, we check whether the metadata basis is strong enough. Coverage is itself a governance issue: if long-tail or non-US films are more likely to have missing metadata, an audit can understate the very problem it is trying to measure.


In [7]:
def count_json_files(path):
    return len(list(path.glob("*.json"))) if path and path.exists() and path.is_dir() else 0

feature_coverage = pd.DataFrame([
    {"feature": "MPNet text matrix", "available": bool(resolved["MPNet matrix"]), "count_or_rows": int(np.load(resolved["MPNet matrix"], mmap_mode="r").shape[0]) if resolved["MPNet matrix"] else np.nan},
    {"feature": "CLIP image matrix", "available": bool(resolved["CLIP image matrix"]), "count_or_rows": int(np.load(resolved["CLIP image matrix"], mmap_mode="r").shape[0]) if resolved["CLIP image matrix"] else np.nan},
    {"feature": "MPNet JSON files", "available": bool(resolved["MPNet JSON folder"]), "count_or_rows": count_json_files(resolved["MPNet JSON folder"])},
    {"feature": "CLIP image JSON files", "available": bool(resolved["CLIP image JSON folder"]), "count_or_rows": count_json_files(resolved["CLIP image JSON folder"])},
])

join_funnel = pd.DataFrame([
    {"step": "MovieLens catalogue", "movies": movies["movieId"].nunique()},
    {"step": "M3L items mapped to MovieLens", "movies": movie_level["movieId"].nunique()},
    {"step": "with IMDb ID", "movies": movie_level["imdb_id_str"].notna().sum()},
    {"step": "with Wikidata match", "movies": movie_level["has_wikidata_match"].sum()},
    {"step": "with country metadata", "movies": movie_level["has_country"].sum()},
    {"step": "with original-language metadata", "movies": movie_level["has_original_language"].sum()},
])
join_funnel["share_of_movielens_catalogue"] = join_funnel["movies"] / max(movies["movieId"].nunique(), 1)

label_summary = pd.DataFrame([
    {"label": "European wide", "movies": int(movie_level["is_european_wide"].sum()), "share": movie_level["is_european_wide"].mean()},
    {"label": "Non-English", "movies": int(movie_level["is_non_english"].fillna(False).sum()), "share": movie_level["is_non_english"].fillna(False).mean()},
    {"label": "US origin", "movies": int(movie_level["is_us_origin"].sum()), "share": movie_level["is_us_origin"].mean()},
    {"label": "US company signal", "movies": int(movie_level["has_us_company_signal"].sum()), "share": movie_level["has_us_company_signal"].mean()},
    {"label": "Long-tail", "movies": int(movie_level["is_long_tail"].sum()), "share": movie_level["is_long_tail"].mean()},
    {"label": "Blockbuster-head", "movies": int(movie_level["is_blockbuster_head"].sum()), "share": movie_level["is_blockbuster_head"].mean()},
])

display(feature_coverage)
display(join_funnel.style.format({"share_of_movielens_catalogue": "{:.1%}"}))
display(label_summary.style.format({"share": "{:.1%}"}))

display(Markdown(
    f"**Interpretation.** The join funnel keeps metadata missingness visible. In this run, "
    f"{int(join_funnel.loc[join_funnel['step'].eq('with country metadata'), 'movies'].iloc[0]):,} mapped M3L movie rows have country metadata and "
    f"{int(join_funnel.loc[join_funnel['step'].eq('with original-language metadata'), 'movies'].iloc[0]):,} have original-language metadata. "
    "These counts define the support behind later country and language charts."
))


/var/folders/64/zkcqc27x1p3cgq_28fymy7700000gn/T/ipykernel_96837/4160804772.py:23: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  {"label": "Non-English", "movies": int(movie_level["is_non_english"].fillna(False).sum()), "share": movie_level["is_non_english"].fillna(False).mean()},


,feature,available,count_or_rows
0,MPNet text matrix,True,19009
1,CLIP image matrix,True,19009
2,MPNet JSON files,True,25177
3,CLIP image JSON files,True,25037


,step,movies,share_of_movielens_catalogue
0,MovieLens catalogue,27278,100.0%
1,M3L items mapped to MovieLens,18973,69.6%
2,with IMDb ID,18973,69.6%
3,with Wikidata match,18684,68.5%
4,with country metadata,18679,68.5%
5,with original-language metadata,18353,67.3%


,label,movies,share
0,European wide,5823,30.7%
1,Non-English,3522,18.6%
2,US origin,12603,66.4%
3,US company signal,7397,39.0%
4,Long-tail,4131,21.8%
5,Blockbuster-head,3799,20.0%


**Interpretation.** The join funnel keeps metadata missingness visible. In this run, 18,679 mapped M3L movie rows have country metadata and 18,353 have original-language metadata. These counts define the support behind later country and language charts.

## 8. Persist Small Foundation Diagnostics

The full movie-level table can be rebuilt locally from the source files above. To keep the public repository licence-clean, we only persist small diagnostic tables from this notebook. Raw MovieLens and M3L files are not redistributed.


In [8]:
source_inventory.to_csv(DATA_PROCESSED / "source_inventory_from_notebook.csv", index=False)
bridge_diagnostics.to_csv(DATA_PROCESSED / "m3l_bridge_diagnostics_from_notebook.csv", index=False)
join_funnel.to_csv(DATA_PROCESSED / "join_funnel_from_notebook.csv", index=False)
feature_coverage.to_csv(DATA_PROCESSED / "feature_coverage_from_notebook.csv", index=False)
label_summary.to_csv(DATA_PROCESSED / "label_summary_from_notebook.csv", index=False)

# A tiny preview is enough for review; the full MovieLens-derived table is rebuilt locally.
movie_level[preview_cols].head(100).to_csv(DATA_PROCESSED / "movie_level_source_preview.csv", index=False)

assert NO_SYNTHETIC_DATA is True
assert len(movie_level) > 0
assert movie_level["has_wikidata_match"].sum() > 0, "Wikidata enrichment is missing. Check cache or SPARQL settings."

print("Data foundation completed from real source files only.")
print("Saved small diagnostic tables under data/processed/.")


Data foundation completed from real source files only.
Saved small diagnostic tables under data/processed/.


## 9. What Later Notebooks Should Use

Later notebooks should read this notebook as the source contract:

- MovieLens and M3L are the main data sources.
- The M3L-to-MovieLens bridge is a reproducible join artifact, not a separate source.
- Wikidata enrichment is produced by the SPARQL logic shown above and cached for reproducibility.
- Country and language labels are proxy labels with missingness and co-production caveats.

This makes the rest of the project easier to defend: when a model result refers to “European”, “non-English”, “Spain-origin” or “US-company signal”, those labels are traceable back to this notebook.
